# Schema evolution


In this notebook we will explore the Schema evolution in pySpark. We can ingest one source file to our delta table, and next ingest another file with different schema,
Spark support that evolution such as add new columns.

## Important tip

when writing your dataframe, make sure to use

``` 
.format("delta") \
.option("mergeSchema", "true") \
``` 


## Set Java Home

In [1]:
import os

# Set Java 17 home directory for Ubuntu
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


## Create 2 json files with different schema

### write first json

In [2]:
data_1 = [

    { 'specie' : 'ant', 'name': 'Ana' },
    { 'specie' : 'bee', 'name': 'Bri' },
]


In [3]:
import json

with open( 'animal_1.json', 'w' ) as file:
   json.dump( data_1, file )


In [4]:
data_2 = [

    { 'specie' : 'cat', 'name': 'Clowy', 'age': 5 },
    { 'specie' : 'dog', 'name': 'Drako', 'age': 4 },
]

In [5]:
data_2

[{'specie': 'cat', 'name': 'Clowy', 'age': 5},
 {'specie': 'dog', 'name': 'Drako', 'age': 4}]

In [6]:
with open( 'animal_2.json', 'w' ) as file:
   json.dump( data_2, file )


## Delta Lake

In [7]:
from IPython.display        import display, HTML

#import findspark
#findspark.init()
import pyspark

#from pyspark.context import SparkContext
from pyspark.sql            import SparkSession
from pyspark.sql.functions  import col, explode, udf
from pyspark.sql.types      import Row, StringType, LongType, ArrayType


In [8]:
pyspark.__version__

'4.2.0'

In [9]:
import sys
print( sys.version )

print( sys.version_info )


3.14.4 (main, Jun 18 2026, 14:25:02) [GCC 15.2.0]
sys.version_info(major=3, minor=14, micro=4, releaselevel='final', serial=0)


spark = ( SparkSession.builder   
        .appName( 'my_schema_evolution' ) 
        .getOrCreate()
        )

In [10]:
! pip freeze | grep pyspark

pyspark==4.2.0


In [11]:
from pyspark.sql import SparkSession

# Initialize Spark with Delta Lake support
spark = SparkSession.builder \
    .appName("SchemaEvolutionExample") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/21 12:20:27 WARN Utils: Your hostname, heartwings, resolves to a loopback address: 127.0.1.1; using 192.168.1.10 instead (on interface wlo1)
26/08/21 12:20:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/art/git/data_engineer/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/art/.ivy2.5.2/cache
The jars for the packages stored in: /home/art/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d6a25524-b404-4ac8-b9a1-0bae514bc4fd;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.log

## Batch Ingestion Function

In [14]:
def ingest_with_schema_evolution(source_path, source_format, target_path):
    """
    Reads data from any standard format and appends it to a Delta table 
    with automatic schema evolution.
    """
    print(f"Reading data from {source_path} using format: {source_format}")
    
    # 1. Read source data (handles csv, json, parquet, avro)
    # Note: 'avro' requires 'org.apache.spark:spark-avro_2.12:<version>' package if used
    if source_format.lower() == "csv":
        df = spark.read.format(source_format).option("header", "true").option("inferSchema", "true").load(source_path)
    else:
        try:
            df = spark.read.format(source_format).load(source_path)
        except Exception as e:
            # Allow multi-line JSON structures (like pretty-printed arrays)
            df = spark.read.format("json").option("multiLine", "true").load(source_path)
        
    # 2. Write to Delta destination with mergeSchema enabled
    print(f"Appending data to destination: {target_path} with schema evolution...")
    df.write \
        .format("delta") \
        .option("mergeSchema", "true") \
        .mode("append") \
        .save(target_path)
    
    print("Ingestion complete.\n")



## explore source files, they have different schema

# Run with Schema evolution

### ingest first.json

In [15]:

# Define target folder
target_table_path = "/home/art/git/data_engineer/spark/animal"

# Step 1: Ingest first file (columns: name, age)
ingest_with_schema_evolution(
    source_path="/home/art/git/data_engineer/spark/animal_1.json", 
    source_format="json", 
    target_path=target_table_path
)


Reading data from /home/art/git/data_engineer/spark/animal_1.json using format: json
Appending data to destination: /home/art/git/data_engineer/spark/animal with schema evolution...


26/08/21 12:59:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Ingestion complete.



### ingest second.json


In [16]:

# Step 2: Ingest second file (columns: name, age, specie)
ingest_with_schema_evolution(
    source_path="/home/art/git/data_engineer/spark/animal_2.json", 
    source_format="json", 
    target_path=target_table_path
)




Reading data from /home/art/git/data_engineer/spark/animal_2.json using format: json
Appending data to destination: /home/art/git/data_engineer/spark/animal with schema evolution...
Ingestion complete.



# verified the merged result

In [17]:
# Step 3: Verify the merged result
print("Resulting Table Data:")
result_df = spark.read.format("delta").load(target_table_path)
result_df.show()

Resulting Table Data:


+---------------+-----+------+----+
|_corrupt_record| name|specie| age|
+---------------+-----+------+----+
|              [| NULL|  NULL|NULL|
|           NULL|Clowy|   cat|   5|
|           NULL|Drako|   dog|   4|
|              ]| NULL|  NULL|NULL|
|              [| NULL|  NULL|NULL|
|           NULL|  Ana|   ant|NULL|
|           NULL|  Bri|   bee|NULL|
|              ]| NULL|  NULL|NULL|
+---------------+-----+------+----+



### links

https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#schema-merging
